In [366]:
"""
This script creates artificial data for a discrete choice problem.
Assume there are three modes of transportation to choose from. Six fixed
variables were designed as significant and five as non-significant.
Seven random variables (five normal, one uniform, one triangular) were
designed as significant.
Three normal variables were correlated.
Two normal variables were non-linearly transformed.
"""


'\nThis script creates artificial data for a discrete choice problem.\nAssume there are three modes of transportation to choose from. Six fixed\nvariables were designed as significant and five as non-significant.\nSeven random variables (five normal, one uniform, one triangular) were\ndesigned as significant.\nThree normal variables were correlated.\nTwo normal variables were non-linearly transformed.\n'

In [367]:

import numpy as np

import random





import statsmodels.api as sm
from scipy.stats import multivariate_normal

In [368]:
import numpy as np
import pandas as pd
import scipy.stats as ss

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
is_correlated = True
if is_correlated:
    name_2 = 'artificial_mixed_corr_2023_MOOF.csv'
else:
    name_2 = 'artificial_ZA.csv'


In [369]:
def noise(n_obs, perc=1, random_state=None):
    random_state = random_state or np.random
    noise_vec = random_state.normal(0, perc, n_obs)
    return noise_vec

In [370]:
def random_col(N, P, J, random_state=np.random, noise_val = 0.25):
    rand_nums = random_state.uniform(low=0, high=1, size=(P,))/100
    return np.tile(rand_nums, N*P) + noise_val*noise(N*P*J, random_state=random_state)

def generate_random_df(N, P, J, num_fixed=0, num_isvars=0, num_randvars=0, random_state=None, add_constant = 1, non_sig_num = None):
    df = pd.DataFrame()
    df['id'] = np.repeat(np.arange(1, (N*P+1)), J)
    #df['ind_id'] = np.repeat(np.arange(1, N+1), P*J)
    #df['alt'] = np.tile(np.arange(1, J+1), N*P)

    varnames = []
    col = random_col(num_fixed, P, J, random_state, 0)
    col = [random.random() for _ in range(num_fixed+num_randvars)]
    col = list(range(1, num_fixed+num_randvars+1))
 
# Shuffle the list
    random.shuffle(col)
    print(col)
    print(col, 'fixed')
    X = (np.random.multivariate_normal(mean=col, cov=np.eye(len(col)), size=N))
    X = np.interp(X, (X.min(), X.max()), (0,1))
    #X = scaler.fit_transform(X)
    print(X.shape)
    for i in range(num_fixed):
        print(X[:,i].shape)
        coef_name = 'fixed' + str(i+1)
        varnames.append(coef_name)
        random_col(N, P, J, random_state=random_state)
        df[coef_name] = random_col(N, P, J, random_state=random_state)
       
        df[coef_name] = X[:,i]
        

    for i in range(num_isvars):
        coef_name = 'added_isvar' + str(i+1)
        varnames.append(coef_name)
        col_vals = np.repeat(random_state.random(N*P)*100, J)
        for j in range(J):
            if j == 0:
                df[coef_name] = col_vals
            else:
                df[coef_name + "." + str(j+1)] = col_vals

    #ol = random_col(num_randvars, P, J, random_state)
    #col = [random.random() for _ in range(num_randvars)]
    #X = (np.random.multivariate_normal(mean=col, cov=np.eye(num_randvars), size=N)+1)/2
    
    for i in range(num_randvars):
        coef_name = 'random' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state)
        df[coef_name] = X[:,i+num_fixed]

    if add_constant:
        df = sm.add_constant(df, prepend=False)
    
    for i in range(non_sig_num):
        coef_name = 'non_sig' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state) + noise(N, 10)
       
    
    
        
        
    return df, varnames


In [371]:

N =2000  # Number of observations
P = 1  # Number of choices per individual
J = 1  # Number of alternatives
num_fixed = 3
num_isvars = 0
num_nonsig = 5
num_randvars = 3
seed = 122
random_state = np.random.RandomState(seed)
np.random.seed(seed)

random.seed(seed)
df, varnames = generate_random_df(N, P, J, num_fixed=num_fixed, num_isvars=num_isvars,
                                  num_randvars=num_randvars, random_state=random_state, non_sig_num = num_nonsig)

# Shift values <-2 for as inverse boxcox transform will be applied





[3, 6, 1, 2, 5, 4]
[3, 6, 1, 2, 5, 4] fixed
(2000, 6)
(2000,)
(2000,)
(2000,)


In [372]:
df.head(10)

,id,fixed1,fixed2,fixed3,random1,random2,random3,const,non_sig1,non_sig2,non_sig3,non_sig4,non_sig5
0,1,0.493877,0.640371,0.249764,0.221502,0.773740,0.415856,1.0,-7.204643,-0.289961,10.898829,-5.527849,-13.785591
1,2,0.332158,0.746364,0.330359,0.286173,0.494773,0.284242,1.0,-19.286245,2.727749,-1.085744,-6.724133,11.287807
2,3,0.428473,0.644556,0.158604,0.389291,0.724957,0.558161,1.0,-12.665115,25.627652,-0.191596,-5.031199,-14.338374
3,4,0.433189,0.814618,0.327103,0.388619,0.536888,0.490748,1.0,-8.071497,-4.164745,-17.632828,-9.957863,-3.722633
4,5,0.505529,0.668609,0.344401,0.318511,0.633265,0.545977,1.0,-8.300516,4.146677,-7.856364,6.645578,13.015042
5,6,0.434346,0.734074,0.453857,0.263203,0.675127,0.541854,1.0,5.679290,12.241097,16.525508,-14.761827,-7.287236
6,7,0.570773,0.684654,0.165958,0.369861,0.613821,0.500896,1.0,6.028600,-3.285003,2.822585,-4.143897,3.070263
7,8,0.455925,0.560683,0.241237,0.374634,0.558959,0.376827,1.0,2.459943,14.579241,-18.617233,12.904458,-16.598645
8,9,0.400491,0.599696,0.112160,0.335335,0.579640,0.617583,1.0,-1.855231,-10.242165,-13.740858,-17.389632,-14.497550
9,10,0.519352,0.752285,0.157701,0.320985,0.541897,0.421214,1.0,1.867784,8.869317,-1.331851,-22.878878,-2.703377


In [373]:
# Define coefficients (betas)
# Fixed betas
fixed_coefs = [ random_state.choice([-1,1, -1]) *random_state.uniform(1, 4) for i in range(num_fixed)]
fixed_coefs = np.array(fixed_coefs)



fixed_coefs = list(fixed_coefs)


In [374]:
print(fixed_coefs)


print(df.head())

[-1.5295681672160175, 2.024250343387329, 1.7688845953368282]
   id    fixed1    fixed2    fixed3   random1   random2   random3  const  \
0   1  0.493877  0.640371  0.249764  0.221502  0.773740  0.415856    1.0   
1   2  0.332158  0.746364  0.330359  0.286173  0.494773  0.284242    1.0   
2   3  0.428473  0.644556  0.158604  0.389291  0.724957  0.558161    1.0   
3   4  0.433189  0.814618  0.327103  0.388619  0.536888  0.490748    1.0   
4   5  0.505529  0.668609  0.344401  0.318511  0.633265  0.545977    1.0   

    non_sig1   non_sig2   non_sig3  non_sig4   non_sig5  
0  -7.204643  -0.289961  10.898829 -5.527849 -13.785591  
1 -19.286245   2.727749  -1.085744 -6.724133  11.287807  
2 -12.665115  25.627652  -0.191596 -5.031199 -14.338374  
3  -8.071497  -4.164745 -17.632828 -9.957863  -3.722633  
4  -8.300516   4.146677  -7.856364  6.645578  13.015042  


In [375]:
# Random mean between -1.5 and 1.5, excluding -.1 - .1 as hard to detect effect
random_coefs_mean = [random_state.choice([-1,1,-1,1, 1, 1]) * random_state.uniform(1, 5) for i in range(num_randvars)]

random_coefs_sd = [random_state.uniform(.5, 3) for i in range(num_randvars)]

cov_mat = np.diag(random_coefs_sd)
if is_correlated:

    #cov_mat = np.diag(random_coefs_sd)

    cov_mat[0, 1] = cov_mat[1, 0] = 0.5
##cov_mat[0, 2] = cov_mat[2, 0] = 0.4
#cov_mat[1, 2] = cov_mat[2, 1] = 0.5



random_coefs_uniform_a = 0
random_coefs_uniform_b = random_state.uniform(1, 2)

random_coefs_tri_left = 0
random_coefs_tri_right = random_state.uniform(1, 2)
random_coefs_tri_mode = random_coefs_tri_right/2


rand_coefs = [np.array([]) for i in range(num_randvars)]

for i in range(N):
    
    res_normal = random_state.multivariate_normal(random_coefs_mean, cov_mat)
    res_normal1= np.array([random_state.normal(random_coefs_uniform_a, random_coefs_uniform_b)])
    res_triangular = np.array([random_state.normal(0, 1)])
    res = np.concatenate((res_normal, np.random.normal(.5,scale = 0.5, size=1), np.random.normal(.5, scale = 1, size=1)))
    res = res_normal
    for r in range(num_randvars):
        rand_coefs[r] = np.append(rand_coefs[r], np.repeat(res[r], P*J))


In [376]:

print(random_coefs_mean)

print(random_coefs_sd)
print(cov_mat)
print(res)

[3.5570807596558414, -4.251208905572314, 4.011107680508342]
[2.682764328965227, 1.5435846652140324, 1.6776999166034292]
[[2.68276433 0.5        0.        ]
 [0.5        1.54358467 0.        ]
 [0.         0.         1.67769992]]
[ 3.72591886 -3.84463938  1.52687271]


In [377]:
random_coefs_uniform_a, random_coefs_uniform_b

(0, 1.9310898036500999)

In [378]:
random_coefs_tri_left, random_coefs_tri_mode, random_coefs_tri_right

(0, 0.5574459482334274, 1.1148918964668548)

In [379]:
B_fixed = [np.repeat(f_coef, P*N*J) for f_coef in fixed_coefs]
B_const = [np.repeat(-2, P*N*J)]



# Convert betas to matrix for easy product
B = [B_fixed, rand_coefs, B_const]
B = [B_i for B_i in B if B_i != []]
B = np.vstack(B).T

In [380]:
# Visualise values after non-linear transformation
# import matplotlib.pyplot as plt
B.shape



(2000, 7)

In [381]:
# Multiply and generate probability
isvars = ['added_isvar' + str(i+1) for i in range(num_isvars)]

X = df.values[:, 1:num_fixed+num_randvars+2]  # Extract only necessary columns
print(X.shape)
XB = (X*B).sum(axis=1).reshape(N*P, J)
eps = np.random.gumbel(0, 1, (N*P, J))
eXB = np.exp(XB).ravel()
dispersion = 1
scale = 1

xg = np.random.gamma(dispersion, scale, N)
#xg = np.array(rgamma(N, dispersion, dispersion))
xbg = eXB
#xbg = eXB

print(max(eXB))
nbyo = np.random.poisson(xbg)

# Use monte carlo simulation to predict choice
# y = np.apply_along_axis(lambda p: np.eye(J, dtype='int64')[np.argmax(p)], 1, prob).reshape(N*P*J,)
# y = y.reshape(N*P*J,)
print('max is', max(nbyo))
y = []
U = XB + eps

for row in U:
    idx = np.argmax(row)
    row_i = np.zeros(J)
    row_i[idx] = 1
    y.append(row_i)
y = np.array(y)
y = np.squeeze(y.reshape(-1, 1))

df['Y'] = nbyo
count = len(list(filter(lambda x: x != 0, nbyo)))
print(count)  # Output: 4

# Apply non-linear transformations for boxcox testing
save = "C:/Users/n9471103/source/repos/HS_BIC/" +name_2
# Save to CSV
print(df.head())
df = df.drop(columns = ['id'])
print(df.head())
df.to_csv(save, index=False)

(2000, 7)
132.65939194631616
max is 147
1228
   id    fixed1    fixed2    fixed3   random1   random2   random3  const  \
0   1  0.493877  0.640371  0.249764  0.221502  0.773740  0.415856    1.0   
1   2  0.332158  0.746364  0.330359  0.286173  0.494773  0.284242    1.0   
2   3  0.428473  0.644556  0.158604  0.389291  0.724957  0.558161    1.0   
3   4  0.433189  0.814618  0.327103  0.388619  0.536888  0.490748    1.0   
4   5  0.505529  0.668609  0.344401  0.318511  0.633265  0.545977    1.0   

    non_sig1   non_sig2   non_sig3  non_sig4   non_sig5  Y  
0  -7.204643  -0.289961  10.898829 -5.527849 -13.785591  0  
1 -19.286245   2.727749  -1.085744 -6.724133  11.287807  3  
2 -12.665115  25.627652  -0.191596 -5.031199 -14.338374  2  
3  -8.071497  -4.164745 -17.632828 -9.957863  -3.722633  5  
4  -8.300516   4.146677  -7.856364  6.645578  13.015042  1  
     fixed1    fixed2    fixed3   random1   random2   random3  const  \
0  0.493877  0.640371  0.249764  0.221502  0.773740  0.41585

In [382]:
# np.linalg.norm(model.coeff_[:11] - fixed_coefs)